# HIV/AIDS Risk Scoring Model
This notebook demonstrates how to build a predictive model for HIV/AIDS risk using survey data. This covers data loading, cleaning, feature engineering, logistic regression modeling, and time-series analysis (if applicable).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import datetime
sns.set(style='whitegrid')

## Load Raw Survey Data

In [ ]:
# Replace with your file path or simulate data
# Example: df = pd.read_csv('hiv_survey_data.csv')
data = {
    'age': [23, 35, 45, np.nan, 22],
    'sex': ['Male', 'Female', 'Female', 'Male', 'Female'],
    'marital_status': ['Single', 'Married', 'Divorced', 'Single', 'Married'],
    'num_partners': [1, 3, 2, 4, 1],
    'condom_use': ['Yes', 'No', 'No', 'Yes', 'Yes'],
    'sti_history': ['No', 'Yes', 'Yes', 'No', 'No'],
    'hiv_status': [0, 1, 1, 0, 0],
    'survey_date': pd.to_datetime(['2020-01-01', '2020-06-01', '2020-06-01', '2021-01-01', '2021-06-01'])
}
df = pd.DataFrame(data)
df.head()

## Data Cleaning and Preprocessing

In [ ]:
# 1. Check how many missing values there are per column
print(df.isnull().sum())

# 2. See the percentage of missing values per column
missing_percentage = df.isnull().mean() * 100
print(missing_percentage)

# 3. Check for any missing data in the entire dataset
print(df.isnull().values.any())  # Returns True if any value is missing

# 4. View only rows that contain missing data
df[df.isnull().any(axis=1)]

# 5. See missing data visually (optional but helpful)
sns.heatmap(df.isnull(), cbar=False, cmap="Reds")
plt.title("Missing Values Heatmap")
plt.show()

# 6. Handle missing values
imputer = SimpleImputer(strategy='mean')
df['age'] = imputer.fit_transform(df[['age']])

In [ ]:
# Encode categorical variables
df_encoded = pd.get_dummies(df, columns=['sex', 'marital_status', 'condom_use', 'sti_history'], drop_first=True)

# Machine learning models (especially those like logistic regression, SVMs, and neural networks) often work better if all features are 
# on a similar scale.
# Feature scaling
scaler = StandardScaler()  # takes a numeric column like age and changes it so it has a mean of 0 and a standard deviation of 1, 
# which helps machine learning models work better.

# Replace the original age values with the new scaled ones; Look at the age column, compute the mean and standard deviation, and 
# then transform all the values into this new standardized form.
df_encoded['age'] = scaler.fit_transform(df_encoded[['age']])

# Final dataset
df_encoded.head()

## Train-Test Split

In [ ]:
X = df_encoded.drop(columns=['hiv_status', 'survey_date'])
y = df_encoded['hiv_status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## Logistic Regression Model

In [ ]:
# model = LogisticRegression()

# When training the model;
# Problem Fix:
# Too few samples (only 3)- Use all data, or get more
# Imbalanced data- Use class_weight='balanced'
model.fit(X_train, y_train)
print(y_train.value_counts())

# When your data is imbalanced — meaning you have way more examples of one class than the other.
# 🔁 Option 1: Use class_weight='balanced' in your model (Gives more attention to the rare class when learning).
model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)

# try to guess the HIV status for the new data you’ve never seen before (X_test)
y_pred = model.predict(X_test)
# tell me how confident you are about each prediction.  [:, 1] means: Give me just the probability of being Positive for each case.
y_prob = model.predict_proba(X_test)[:, 1]

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print('AUC-ROC:', roc_auc_score(y_test, y_prob))

# 📌 Summary:
# "I told Python to use logistic regression to guess HIV status."
# "I trained it on 70% of the data so it could learn patterns."
# "Then I tested it on the remaining 30% to see how good it was."
# "I looked at the guesses it made, how confident it was, and how often it was right or wrong using accuracy, precision, recall, F1, and AUC."

In [ ]:
# 🧰 Option 2: Upsample the minority class (make more ‘Positive’ examples)
from sklearn.utils import resample

# Combine X and y
train_data = pd.concat([X_train, y_train], axis=1)

# Separate classes
majority = train_data[train_data.hiv_status == 'Negative']
minority = train_data[train_data.hiv_status == 'Positive']

# Upsample minority
minority_upsampled = resample(minority,
                              replace=True,
                              n_samples=len(majority),
                              random_state=42)

# Combine back
upsampled = pd.concat([majority, minority_upsampled])

# Split again
X_train = upsampled.drop('hiv_status', axis=1)
y_train = upsampled['hiv_status']

## Time-Series Analysis of HIV Trends

In [ ]:
df['year_month'] = df['survey_date'].dt.to_period('M')
time_series = df.groupby('year_month')['hiv_status'].mean()
time_series.plot(kind='line', marker='o', title='Monthly HIV Positivity Rate')
plt.ylabel('HIV+ Rate')
plt.xlabel('Month')
plt.tight_layout()
plt.show()

## Next Steps
- Apply more complex ML models (e.g., Random Forest, XGBoost)
- Use SMOTE for class balancing
- Conduct SHAP analysis for model explainability
- Integrate model into dashboard (Power BI, Streamlit)

In [ ]:
print(False + False)
print(True + False)
print(False + True)
print(True + True)
print(False + True + True + True)

In [ ]:
# Check your answer (Run this code cell to receive credit!)
q4.check()